In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp plotting

# Plotting

> Time series visualizations

In [ ]:
#| hide
from nbdev.showdoc import show_doc

In [ ]:
#| export
import random
import re
from itertools import product
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import matplotlib.colors as cm
import numpy as np
import pandas as pd

from utilsforecast.compat import DataFrame, pl_DataFrame
from utilsforecast.processing import DataFrameProcessing

In [ ]:
#| exporti
def _parse_ds_type(df: pd.DataFrame):
    dt_col = df["ds"]
    dt_check = pd.api.types.is_datetime64_any_dtype(dt_col)
    int_float_check = dt_col.dtype.kind in ["i", "f"]
    if not dt_check and not int_float_check:
        df = df.copy()
        try:
            df["ds"] = pd.to_datetime(df["ds"])
        except Exception as e:
            msg = (
                "Failed to parse `ds` column as datetime. "
                "Please use `pd.to_datetime` outside to fix the error. "
                f"{e}"
            )
            raise Exception(msg) from e
    return df

In [ ]:
#| export
def plot(
    df: DataFrame,
    forecasts_df: Optional[DataFrame] = None,
    unique_ids: Optional[List[str]] = None,
    plot_random: bool = True,
    models: Optional[List[str]] = None,
    level: Optional[List[float]] = None,
    max_insample_length: Optional[int] = None,
    plot_anomalies: bool = False,
    engine: str = "matplotlib",
    resampler_kwargs: Optional[Dict] = None,
):
    """Plot forecasts and insample values.

    Parameters
    ----------
    df : pandas or polars DataFrame
        DataFrame with columns [`unique_id`, `ds`, `y`].
    forecasts_df : pandas or polars DataFrame, optional (default=None)
        DataFrame with columns [`unique_id`, `ds`] and models.
    unique_ids : list of str, optional (default=None)
        Time Series to plot.
        If None, time series are selected randomly.
    plot_random : bool (default=True)
        Select time series to plot randomly.
    models : list of str, optional (default=None)
        Models to plot.
    level : list of float, optional (default=None)
        Prediction intervals to plot.
    max_insample_length : int, optional (default=None)
        Maximum number of train/insample observations to be plotted.
    plot_anomalies : bool (default=False)
        Plot anomalies for each prediction interval.
    engine : str (default='matplotlib')
        Library used to plot. 'plotly', 'plotly-resampler' or 'matplotlib'.
    resampler_kwargs : dict
        Keyword arguments to be passed to plotly-resampler constructor.
        For further custumization ("show_dash") call the method,
        store the plotting object and add the extra arguments to
        its `show_dash` method.

    Returns
    -------
    fig : matplotlib or plotly figure
        Plot's figure
    """
    if level is not None and not isinstance(level, list):
        raise Exception(
            "Please use a list for the `level` argument "
            "If you only have one level, use `level=[your_level]`"
        )

    if isinstance(df, pl_DataFrame):
        df = df.to_pandas()
    if isinstance(forecasts_df, pl_DataFrame):
        forecasts_df = forecasts_df.to_pandas()

    if unique_ids is None:
        df_pt = DataFrameProcessing(
            dataframe=df, sort_dataframe=True, validate=False
        )
        uids_arr: pd.Index = df_pt.indices
        uid_dtype = uids_arr.dtype

        if df.index.name != "unique_id":
            df["unique_id"] = df["unique_id"].astype(uid_dtype)
            df = df.set_index("unique_id")
        else:
            df.index = df.index.astype(uid_dtype)

        if forecasts_df is not None:
            if isinstance(forecasts_df, pl_DataFrame):
                forecasts_df = forecasts_df.to_pandas()

            if forecasts_df.index.name == "unique_id":
                forecasts_df.index = forecasts_df.index.astype(uid_dtype)
                unique_ids = np.intersect1d(uids_arr, forecasts_df.index.unique())
            else:
                forecasts_df["unique_id"] = forecasts_df["unique_id"].astype(
                    uid_dtype
                )
                unique_ids = np.intersect1d(
                    uids_arr, forecasts_df["unique_id"].unique()
                )
        else:
            unique_ids = uids_arr
    
    if plot_random:
        unique_ids = random.sample(list(unique_ids), k=min(8, len(unique_ids)))
    else:
        unique_ids = unique_ids[:8]
    n_series = len(unique_ids)

    if engine in ["plotly", "plotly-resampler"]:
        try:
            import plotly.graph_objects as go
            from plotly.subplots import make_subplots
        except ImportError:
            raise ImportError(
                "plotly is not installed. "
                "Please install it with `pip install utilsforecast[plotly]`"
            )
        n_rows = min(4, len(unique_ids) // 2 + 1 if len(unique_ids) > 2 else 1)
        fig = make_subplots(
            rows=n_rows,
            cols=2 if len(unique_ids) >= 2 else 1,
            vertical_spacing=0.1,
            horizontal_spacing=0.07,
            x_title="Datestamp [ds]",
            y_title="Target [y]",
            subplot_titles=[str(uid) for uid in unique_ids],
        )
        if engine == "plotly-resampler":
            try:
                from plotly_resampler import FigureResampler
            except ImportError:
                raise ImportError(
                    "plotly-resampler is not installed. "
                    "Please install it with `pip install utilsforecast[plotly-resampler]`"
                )
            resampler_kwargs = {} if resampler_kwargs is None else resampler_kwargs
            fig = FigureResampler(fig, **resampler_kwargs)
        showed_legends: set = set()

        def plotly(
            df,
            fig,
            n_rows,
            unique_ids,
            models,
            plot_anomalies,
            max_insample_length,
            showed_legends,
        ):
            if models is None:
                exclude_str = ["lo", "hi", "unique_id", "ds"]
                models = [
                    c
                    for c in df.columns
                    if all(item not in c for item in exclude_str)
                ]
            if "y" not in models:
                models = ["y"] + models
            for uid, (idx, idy) in zip(
                unique_ids, product(range(1, n_rows + 1), range(1, 2 + 1))
            ):
                df_uid = df.query("unique_id == @uid")
                if max_insample_length:
                    df_uid = df_uid.iloc[-max_insample_length:]
                plot_anomalies = "y" in df_uid and plot_anomalies
                df_uid = _parse_ds_type(df_uid)
                colors = plt.cm.get_cmap("tab20b", len(models))
                colors = ["#1f77b4"] + [
                    cm.to_hex(colors(i)) for i in range(len(models))
                ]
                for col, color in zip(models, colors):
                    if col in df_uid:
                        model = df_uid[col]
                        fig.add_trace(
                            go.Scatter(
                                x=df_uid["ds"],
                                y=model,
                                mode="lines",
                                name=col,
                                legendgroup=col,
                                line=dict(color=color, width=1),
                                showlegend=(
                                    idx == 1
                                    and idy == 1
                                    and col not in showed_legends
                                ),
                            ),
                            row=idx,
                            col=idy,
                        )
                        showed_legends.add(col)
                    model_has_level = any(f"{col}-lo" in c for c in df_uid)
                    if level is not None and model_has_level:
                        level_ = level
                    elif model_has_level:
                        level_col = df_uid.filter(like=f"{col}-lo").columns[0]
                        level_col = re.findall(
                            "[\d]+[.,\d]+|[\d]*[.][\d]+|[\d]+", level_col
                        )[0]
                        level_ = [level_col]
                    else:
                        level_ = []
                    ds = df_uid["ds"]
                    for lv in level_:
                        lo = df_uid[f"{col}-lo-{lv}"]
                        hi = df_uid[f"{col}-hi-{lv}"]
                        plot_name = f"{col}_level_{lv}"
                        fig.add_trace(
                            go.Scatter(
                                x=np.concatenate([ds, ds[::-1]]),
                                y=np.concatenate([hi, lo[::-1]]),
                                fill="toself",
                                mode="lines",
                                fillcolor=color,
                                opacity=-float(lv) / 100 + 1,
                                name=plot_name,
                                legendgroup=plot_name,
                                line=dict(color=color, width=1),
                                showlegend=(
                                    idx == 1
                                    and idy == 1
                                    and plot_name not in showed_legends
                                ),
                            ),
                            row=idx,
                            col=idy,
                        )
                        showed_legends.add(plot_name)
                        if col != "y" and plot_anomalies:
                            anomalies = (df_uid["y"] < lo) | (df_uid["y"] > hi)
                            plot_name = f"{col}_anomalies_level_{lv}"
                            fig.add_trace(
                                go.Scatter(
                                    x=ds[anomalies],
                                    y=df_uid["y"][anomalies],
                                    fillcolor=color,
                                    mode="markers",
                                    opacity=float(lv) / 100,
                                    name=plot_name,
                                    legendgroup=plot_name,
                                    line=dict(color=color, width=0.7),
                                    marker=dict(
                                        size=4, line=dict(color="red", width=0.5)
                                    ),
                                    showlegend=(
                                        idx == 1
                                        and idy == 1
                                        and plot_name not in showed_legends
                                    ),
                                ),
                                row=idx,
                                col=idy,
                            )
                            showed_legends.add(plot_name)
            return fig

        fig = plotly(
            df=df,
            fig=fig,
            n_rows=n_rows,
            unique_ids=unique_ids,
            models=models,
            plot_anomalies=plot_anomalies,
            max_insample_length=max_insample_length,
            showed_legends=showed_legends,
        )
        if forecasts_df is not None:
            fig = plotly(
                df=forecasts_df,
                fig=fig,
                n_rows=n_rows,
                unique_ids=unique_ids,
                models=models,
                plot_anomalies=plot_anomalies,
                max_insample_length=None,
                showed_legends=showed_legends,
            )
        fig.update_xaxes(matches=None, showticklabels=True, visible=True)
        fig.update_layout(margin=dict(l=60, r=10, t=20, b=50))
        fig.update_layout(template="plotly_white", font=dict(size=10))
        fig.update_annotations(font_size=10)
        fig.update_layout(autosize=True, height=150 * n_rows)

    elif engine == "matplotlib":
        if len(unique_ids) == 1:
            n_cols = 1            
        else:
            n_cols = 2
        quot, resid = divmod(n_series, n_cols)
        n_rows = min(4, quot + resid)
        fig, ax = plt.subplots(
            nrows=n_rows,
            ncols=n_cols,
            figsize=(24, 3.5 * n_rows),
            squeeze=False,
            gridspec_kw=dict(hspace=0.5),
        )

        for uid, axi in zip(unique_ids, ax.flat):
            train_uid = df.query("unique_id == @uid")
            train_uid = _parse_ds_type(train_uid)
            if max_insample_length is not None:
                train_uid = train_uid.iloc[-max_insample_length:]
            ds = train_uid["ds"]
            y = train_uid["y"]
            axi.plot(ds, y, label="y")                
            if forecasts_df is not None:
                if models is None:
                    exclude_str = ["lo", "hi", "unique_id", "ds"]
                    models = [
                        c
                        for c in forecasts_df.columns
                        if all(item not in c for item in exclude_str)
                    ]
                if "y" not in models:
                    models = ["y"] + models
                test_uid = forecasts_df.query("unique_id == @uid")
                plot_anomalies = "y" in test_uid and plot_anomalies
                test_uid = _parse_ds_type(test_uid)
                first_ds_fcst = test_uid["ds"].min()
                axi.axvline(
                    x=first_ds_fcst,
                    color="black",
                    label="First ds Forecast",
                    linestyle="--",
                )
                colors = plt.cm.get_cmap("tab20b", len(models))
                colors = ["blue"] + [colors(i) for i in range(len(models))]
                for col, color in zip(models, colors):
                    if col in test_uid:
                        axi.plot(
                            test_uid["ds"], test_uid[col], label=col, color=color
                        )
                    model_has_level = any(f"{col}-lo" in c for c in test_uid)
                    if level is not None and model_has_level:
                        level_ = level
                    elif model_has_level:
                        level_col = test_uid.filter(like=f"{col}-lo").columns[0]
                        level_col = re.findall(
                            "[\d]+[.,\d]+|[\d]*[.][\d]+|[\d]+", level_col
                        )[0]
                        level_ = [level_col]
                    else:
                        level_ = []
                    for lv in level_:
                        ds_test = test_uid["ds"]
                        lo = test_uid[f"{col}-lo-{lv}"]
                        hi = test_uid[f"{col}-hi-{lv}"]
                        axi.fill_between(
                            ds_test,
                            lo,
                            hi,
                            alpha=-float(lv) / 100 + 1,
                            color=color,
                            label=f"{col}_level_{lv}",
                        )
                        if col != "y" and plot_anomalies:
                            anomalies = (test_uid["y"] < lo) | (test_uid["y"] > hi)
                            axi.scatter(
                                x=ds_test[anomalies],
                                y=test_uid["y"][anomalies],
                                color=color,
                                s=30,
                                alpha=float(lv) / 100,
                                label=f"{col}_anomalies_level_{lv}",
                                linewidths=0.5,
                                edgecolors="red",
                            )

            axi.set_title(f"{uid}")
            axi.set_xlabel("Datestamp [ds]")
            axi.set_ylabel("Target [y]")
            axi.legend(loc="upper left")
            axi.xaxis.set_major_locator(plt.MaxNLocator())
            axi.grid()
        plt.close(fig)
        if len(ax.flat) > n_series:
            for axi in ax.flat[n_series:]:
                axi.set_axis_off()
    else:
        raise Exception(f"Unkwown plot engine {engine}")
    return fig

In [ ]:
show_doc(plot)

In [ ]:
from utilsforecast.data import generate_series

In [ ]:
series = generate_series(4, freq='D', equal_ends=True, with_trend=True)
test = series.groupby('unique_id').tail(10).copy()
train = series.drop(test.index)
for i in range(4):
    test[f'model{i}'] = test['y'] * (1 + 0.1 * np.random.rand(test.shape[0]))

In [ ]:
plot(
    series,
    forecasts_df=test,
    plot_random=False,
    max_insample_length=50,
    engine='matplotlib',
)

In [ ]:
plot(
    series,
    forecasts_df=test,
    plot_random=False,
    max_insample_length=50,
    engine='plotly',
)